In [ ]:
%matplotlib widget

import sys
from typing import Optional, Tuple

import matplotlib.pyplot as plt
import MRzeroCore as mr0
import numpy as np
from numpy import pi
import time
import torch.nn as nn
import pypulseq as pp
import torch
from Modules.CoilSens import cyclic_head_coil
from Modules.FourierTransform import (
    nonuniform_fourier_transform_adjoint,
    nonuniform_fourier_transform_forward,
)
from Modules.Linops import ChannelOperator, LinearOperator, IdentityOperator
from Modules.NumSolvers import conjugate_gradient
import Modules.ConcatSum
from Modules.Plan import KbNufft, KbNufftAdjoint
from tqdm import tqdm
from Modules.Regularizer import Regularizer
from Modules.fast_iso import isochromat_sim_fast

sys.path.insert(0, "..")
from Modules import default_system

torch.set_default_device("cuda")

In [ ]:
class NonuniformFourierTransformOperator(LinearOperator):

    def __init__(
        self,
        k: torch.Tensor,
        input_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ):
        nD, nK, nTI = k.shape
        nC, N_x, nY, TI = input_shape

        self.n_modes = (N_x, nY)
        self.k = k
        self.forward_shape = (nC, N_x, nY, 1, nTI)
        self.adjoint_shape = (nC, nK, nTI)
        self.shape = (
            int(2 * torch.prod(torch.tensor(self.adjoint_shape))),
            int(2 * torch.prod(torch.tensor(self.forward_shape))),
        )
        self.nufft_ob = KbNufft(im_size=self.n_modes).to(device)
        self.adj_ob = KbNufftAdjoint(im_size=self.n_modes).to(device)

        self.dtype = torch.float32
        self.device = device

    def _matvec(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.forward_shape)
        y = nonuniform_fourier_transform_forward(self.k, x, nufft_ob=self.nufft_ob)
        return torch.view_as_real(y.reshape(-1)).reshape(-1)

    def _rmatvec(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.adjoint_shape)
        y = nonuniform_fourier_transform_adjoint(
            self.k, x, self.n_modes, adj_ob=self.adj_ob
        )
        return torch.view_as_real(y.reshape(-1)).reshape(-1)


class ChannelOperator(LinearOperator):

    def __init__(
        self,
        coil_sensitivities: torch.Tensor,
        data_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ):
        nC = coil_sensitivities.shape[0]

        self.coil_sensitivities = coil_sensitivities
        self.device = device
        self.dtype = torch.float32
        self.internal_dtype = torch.complex64
        self.forward_shape = (1, *data_shape)
        self.adjoint_shape = (nC, *data_shape)
        self.shape = (
            int(2 * torch.prod(torch.tensor(self.adjoint_shape))),
            int(2 * torch.prod(torch.tensor(self.forward_shape))),
        )

    def _matvec(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.forward_shape)
        y = self.coil_sensitivities * x
        return torch.view_as_real(y.reshape(-1)).reshape(-1)

    def _rmatvec(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.adjoint_shape)
        y = torch.sum(torch.conj(self.coil_sensitivities) * x, dim=0, keepdim=True)
        return torch.view_as_real(y.reshape(-1)).reshape(-1)


def conjugate_gradient(
    Q: LinearOperator,
    b: torch.Tensor,
    x: torch.Tensor,
    maxiter: int = 10,
    disable_progressbar: bool = False,
) -> torch.Tensor:

    g_prev = torch.ones(x.shape, dtype=x.dtype)
    p_prev = torch.zeros(x.shape, dtype=x.dtype)

    for iteration in tqdm(range(maxiter), disable=disable_progressbar):

        g = Q @ x - b

        conjugacy_factor = g @ g / (g_prev @ g_prev)
        p = g + conjugacy_factor * p_prev
        stepsize = g @ g / (p @ (Q @ p))
        x = x - stepsize * p

        g_prev = g.clone()
        p_prev = p.clone()

    return x

In [ ]:
class NonuniformFourierTransformOperator(LinearOperator):

    def __init__(
        self,
        k: torch.Tensor,
        input_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ):
        nD, nK, nTI = k.shape
        nC, N_x, nY, TI = input_shape

        self.n_modes = (N_x, nY)
        self.k = k
        self.forward_shape = (nC, N_x, nY, 1, nTI)
        self.adjoint_shape = (nC, nK, nTI)
        self.shape = (
            int(2 * torch.prod(torch.tensor(self.adjoint_shape))),
            int(2 * torch.prod(torch.tensor(self.forward_shape))),
        )
        self.nufft_ob = KbNufft(im_size=self.n_modes).to(device)
        self.adj_ob = KbNufftAdjoint(im_size=self.n_modes).to(device)

        self.dtype = torch.float32
        self.device = device

    def _matvec(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.forward_shape)
        y = nonuniform_fourier_transform_forward(self.k, x, nufft_ob=self.nufft_ob)
        return torch.view_as_real(y.reshape(-1)).reshape(-1)

    def _rmatvec(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.adjoint_shape)
        y = nonuniform_fourier_transform_adjoint(
            self.k, x, self.n_modes, adj_ob=self.adj_ob
        )
        return torch.view_as_real(y.reshape(-1)).reshape(-1)


class ChannelOperator(LinearOperator):

    def __init__(
        self,
        coil_sensitivities: torch.Tensor,
        data_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ):
        nC = coil_sensitivities.shape[0]

        self.coil_sensitivities = coil_sensitivities
        self.device = device
        self.dtype = torch.float32
        self.internal_dtype = torch.complex64
        self.forward_shape = (1, *data_shape)
        self.adjoint_shape = (nC, *data_shape)
        self.shape = (
            int(2 * torch.prod(torch.tensor(self.adjoint_shape))),
            int(2 * torch.prod(torch.tensor(self.forward_shape))),
        )

    def _matvec(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.forward_shape)
        y = self.coil_sensitivities * x
        return torch.view_as_real(y.reshape(-1)).reshape(-1)

    def _rmatvec(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.adjoint_shape)
        y = torch.sum(torch.conj(self.coil_sensitivities) * x, dim=0, keepdim=True)
        return torch.view_as_real(y.reshape(-1)).reshape(-1)


def conjugate_gradient(
    Q: LinearOperator,
    b: torch.Tensor,
    x: torch.Tensor,
    maxiter: int = 10,
    disable_progressbar: bool = False,
) -> torch.Tensor:

    g_prev = torch.ones(x.shape, dtype=x.dtype)
    p_prev = torch.zeros(x.shape, dtype=x.dtype)

    for iteration in tqdm(range(maxiter), disable=disable_progressbar):

        g = Q @ x - b

        conjugacy_factor = g @ g / (g_prev @ g_prev)
        p = g + conjugacy_factor * p_prev
        stepsize = g @ g / (p @ (Q @ p))
        x = x - stepsize * p

        g_prev = g.clone()
        p_prev = p.clone()

    return x

In [ ]:
def truefisp_seq(
    FOV: float = 200e-3,
    N_x: int = 128,
    alpha: float = 60.0,
    slice_thickness: float = 5e-3,
    TRF: float = 0.8e-3,
    TBW: float = 2.0,
    BW: float = 890.0,
    pre_dur: float = 0.6e-3,
    TR: float = 3.12e-3,
    N_readout: int = 1800,
    N_spokes: int = 1,
    R=1,
    N_dummy: int = 0,
    TI_start: float = 20e-3,
    TI_inter: float = 0.0,
    TD: float = 2000e-3,
    write_seq: bool = False,
    plot: bool = False,
) -> pp.Sequence:

    system = default_system()
    seq = pp.Sequence(system)
    deltak = 1 / FOV
    _gr = seq.grad_raster_time

    rf_inv, gz_inv, _ = pp.make_sinc_pulse(
        flip_angle=np.deg2rad(180),
        duration=10e-3,
        slice_thickness=slice_thickness,
        apodization=0.5,
        time_bw_product=4,
        system=system,
        return_gz=True,
        use="inversion",
    )

    rf, gz, _ = pp.make_sinc_pulse(
        apodization=0.5,
        duration=2e-3,
        flip_angle=np.deg2rad(alpha),
        slice_thickness=slice_thickness,
        system=system,
        time_bw_product=4,
        return_gz=True,
        use="excitation",
    )

    _astep = N_x * seq.adc_raster_time
    _k = 1
    while abs(_k * _gr - round(_k * _gr / _astep) * _astep) > 1e-9:
        _k += 1
    readout_flat = int(round((1.0 / BW) / (_k * _gr))) * (_k * _gr)

    gx_base = pp.make_trapezoid(
        channel="x", flat_area=N_x * deltak, flat_time=readout_flat, system=system
    )
    adc = pp.make_adc(
        num_samples=N_x,
        duration=gx_base.flat_time,
        delay=gx_base.rise_time,
        system=system,
    )

    pre_dur = int(np.ceil(pre_dur / _gr)) * _gr
    gx_pre_base = pp.make_trapezoid(
        channel="x", area=-gx_base.area / 2, duration=pre_dur, system=system
    )
    gz_reph = pp.make_trapezoid(
        channel="z", area=-gz.area / 2, duration=pre_dur, system=system
    )

    _fixed = (
        TRF
        + 2 * pre_dur
        + gx_base.flat_time
        + gx_base.rise_time
        + gx_base.fall_time
        + 2 * (gx_pre_base.rise_time + gx_pre_base.fall_time)
    )
    delay_TE = int(np.ceil(max(1e-4, TR - _fixed) / _gr)) * _gr
    TE = pre_dur + TRF / 2 + delay_TE + gx_base.flat_time / 2
    inv_duration = pp.calc_duration(rf_inv, gz_inv)
    delay_TI_start = (
        np.ceil(
            (TI_start - inv_duration / 2 - pp.calc_duration(gz) / 2)
            / seq.grad_raster_time
        )
        * seq.grad_raster_time
    )
    assert delay_TI_start > 0, "TI_start too short"

    gre_block_dur = (
        pp.calc_duration(gx_pre_base)
        + pp.calc_duration(gz)
        + delay_TE
        + pp.calc_duration(gx_base)  # readout
        + pp.calc_duration(gx_pre_base)  # in-plane rewind
    )
    delay_TD = np.ceil(TD / seq.grad_raster_time) * seq.grad_raster_time
    delay_TI_inter = np.ceil(TI_inter / seq.grad_raster_time) * seq.grad_raster_time

    GA_DEG = 111.246
    rf_phase = 0.0
    rf_inc = 180.0
    spoke_counter = 0

    for seg in range(N_spokes // R):
        seq.add_block(rf_inv, gz_inv)
        seq.add_block(pp.make_delay(delay_TI_start))

        n_this_seg = N_readout + (N_dummy if seg == 0 else 0)
        for s in range(n_this_seg):
            is_dummy = (seg == 0) and (s < N_dummy)

            rf.phase_offset = rf_phase / 180 * np.pi
            adc.phase_offset = rf_phase / 180 * np.pi
            rf_phase = (rf_phase + rf_inc) % 360

            if is_dummy:
                angle = 0.0
            else:
                angle = np.deg2rad(spoke_counter * GA_DEG)
                spoke_counter += 1

            kx = np.cos(angle)
            ky = np.sin(angle)

            gx_pre = pp.make_trapezoid(
                channel="x", area=gx_pre_base.area * kx, duration=pre_dur, system=system
            )
            gy_pre = pp.make_trapezoid(
                channel="y", area=gx_pre_base.area * ky, duration=pre_dur, system=system
            )

            gx_rew = pp.make_trapezoid(
                channel="x", area=gx_pre_base.area * kx, duration=pre_dur, system=system
            )
            gy_rew = pp.make_trapezoid(
                channel="y", area=gx_pre_base.area * ky, duration=pre_dur, system=system
            )

            seq.add_block(rf, gz)
            seq.add_block(gx_pre, gy_pre, gz_reph)
            seq.add_block(pp.make_delay(delay_TE))

            gx = pp.make_trapezoid(
                channel="x",
                amplitude=gx_base.amplitude * kx,
                flat_time=gx_base.flat_time,
                rise_time=gx_base.rise_time,
                fall_time=gx_base.fall_time,
                system=system,
            )
            gy = pp.make_trapezoid(
                channel="y",
                amplitude=gx_base.amplitude * ky,
                flat_time=gx_base.flat_time,
                rise_time=gx_base.rise_time,
                fall_time=gx_base.fall_time,
                system=system,
            )

            if is_dummy:
                seq.add_block(gx, gy)
            else:
                seq.add_block(gx, gy, adc)

            seq.add_block(gx_rew, gy_rew, gz_reph)

            if delay_TI_inter > 0 and s < n_this_seg - 1:
                seq.add_block(pp.make_delay(delay_TI_inter))

        seq.add_block(pp.make_delay(TD))

    ok, err = seq.check_timing()
    if ok:
        print("2-D TrueFISP timing check passed")
        print(f"  GRE block dur (TR) : {gre_block_dur * 1000:.2f} ms")
        print(f"  Readout train      : {N_readout * gre_block_dur * 1000:.0f} ms")
        est = N_spokes * (
            inv_duration + delay_TI_start + N_readout * gre_block_dur + TD
        )
        print(f"  Scan time est       : {est / 60:.1f} min")
    else:
        print(f"Timing FAILED: {err}")

    seq.set_definition("FOV", [FOV, FOV, slice_thickness])
    seq.set_definition("Name", "IR_TrueFISP_bssfp.seq")
    if write_seq:
        seq.write("IR_TrueFISP_bssfp.seq")
    if plot:
        seq.plot(time_range=(0.2, 0.225))
    return seq, gre_block_dur, TI_start

In [ ]:
device = torch.device("cuda")
N_x = 128
N_channels = 8
FOV = 200e-3
slice_thickness = 5e-3
alpha = 60.0  # excitation flip angle
TRF = 0.8e-3  # RF pulse duration (TRF/TR = 20%)
TBW = 2.0  # sinc time-bandwidth product
BW = 890.0  # readout bandwidth (Hz/px)
TR = 3.12e-3  # repetition time
pre_dur = 0.6e-3  # pre-scan / rewind gradient duration
N_readout = 1800
N_spokes = 1
R = 1
N_dummy = 0  # sample right after the inversion
TI_start = 20e-3
TI_inter = 0.0
TD = 2000e-3

In [ ]:
seq, gre_block_dur, TI_start = truefisp_seq(
    FOV=FOV,
    N_x=N_x,
    slice_thickness=slice_thickness,
    alpha=alpha,
    TRF=TRF,
    TBW=TBW,
    BW=BW,
    TR=TR,
    pre_dur=pre_dur,
    N_dummy=N_dummy,
    write_seq=True,
    plot=False,
)

In [ ]:
print(f"TR requested: {TR*1e3:.3f} ms   TR actual (gre_block_dur): {gre_block_dur*1e3:.3f} ms   ratio: {gre_block_dur/TR:.3f}")

In [ ]:
phantom = mr0.util.load_phantom(size=(N_x, N_x))

coil_sensitivities_recon = cyclic_head_coil((N_channels, N_x, N_x, 1)).to(
    torch.complex64
)
coil_sensitivities_recon = coil_sensitivities_recon.flip((1, 2))
coil_sensitivities_recon = coil_sensitivities_recon[..., 0]

coil_sensitivities_sim = cyclic_head_coil(
    (N_channels, N_x, N_x, 1),
    fov_scaling=(phantom.size[0] / FOV, phantom.size[1] / FOV, 1),
).to(torch.complex64)
phantom.coil_sens = coil_sensitivities_sim.to(device)
phantom.B0 = 0 * phantom.B0
phantom.B1 = torch.ones(phantom.B1.shape, dtype=phantom.B1.dtype)

data = phantom.build()

In [ ]:
# Simulate the sequence (ground-truth time-domain spin simulation)
mr0_seq = mr0.Sequence.import_file("IR_TrueFISP_bssfp.seq")
torch.manual_seed(0)
signal = isochromat_sim_fast(
    mr0_seq, data.cuda(), spin_count=256, spin_dist="rand", print_progress=True,
)
signal = signal.permute((1, 0)).contiguous()   # (N_channels, N_samples)
print(f"simulated signal: {tuple(signal.shape)}")
torch.save(signal.cpu(), 'IR_trueFISP_bssfp_signal.pt')

In [ ]:
kspace = seq.calculate_kspace()
k_traj_adc = kspace[0]
k_traj_adc = k_traj_adc[:2, :]
k_traj_radius = np.sqrt(np.sum(k_traj_adc**2, axis=0, keepdims=True))
k_traj_adc = k_traj_adc / k_traj_radius.max() * 0.5
k_traj_adc = torch.tensor(k_traj_adc, dtype=torch.float32)

In [ ]:
N_center = 8  # projections for the k-space center
N_max = 55  # outermost Fibonacci ring; also the frame time window
frame_advance = 4  # filter advance per frame (paper: 8 -> 32 ms; 4 -> 16 ms
#   for finer temporal resolution of the IR evolution)
use_dcf = False  # NOT needed: iterative recon models the exact sampling (gridding-only correction)

FIB = [8, 13, 21, 34, 55, 89, 144, 233, 377, 610]


def _next_fib(x):
    return next(f for f in FIB if f >= x)


n_samples_total = k_traj_adc.shape[1]
assert n_samples_total % N_x == 0, "trajectory is not a whole number of spokes"
n_spokes_total = n_samples_total // N_x
assert n_spokes_total > N_max, "need more spokes than N_max for view-sharing"

# radius of each radial sample (0 at center -> 1 at the edge) and its ring count
r = np.linspace(0.0, 1.0, N_x)
ring = np.array([_next_fib(max(N_center, int(np.ceil(N_max * ri)))) for ri in r])
# Ram-Lak 1/r DCF (center clamped to the first non-zero radius)
dcf = np.where(r > 0, 1.0 / r, 1.0 / r[1])

# butterfly offsets: for sample i, the ring[i] projections centered on the frame
djs, is_ = [], []
for i in range(N_x):
    nproj = int(ring[i])
    lo = -(nproj // 2)
    djs.extend(range(lo, lo + nproj))
    is_.extend([i] * nproj)
dj = np.asarray(djs, dtype=np.int64)  # (K,) relative projection offset
ii = np.asarray(is_, dtype=np.int64)  # (K,) radial sample index
K = dj.size
w = dcf[ii] if use_dcf else np.ones(K)  # (K,) DCF per butterfly point

# valid frames: the outermost window [n - N_max//2, n + N_max//2] must fit
n0 = N_max // 2
n_frames = (n_spokes_total - N_max - 1) // frame_advance + 1
centers = n0 + frame_advance * np.arange(n_frames)

# per-frame flat k-space sample indices for the butterfly (n_frames, K)
idx = (centers[:, None] + dj[None, :]) * N_x + ii[None, :]

# Butterfly gather on the CPU (robust for the complex signal), then move the
# result back to the accelerator. flat index is frame-major (frame, point).
sig_cpu = signal.cpu().numpy()  # (C, S) complex64
k_cpu = k_traj_adc.cpu().numpy()  # (2, S) float32
flat = idx.reshape(-1)  # (n_frames * K,)
k_traj_adc_reshaped = (
    torch.from_numpy(k_cpu[:, flat].reshape(2, n_frames, K).transpose(0, 2, 1).copy())
    .float()
    .to(k_traj_adc.device)
)  # (2, K, n_frames)
data_reshaped = torch.from_numpy(
    (
        sig_cpu[:, flat].reshape(N_channels, n_frames, K).transpose(0, 2, 1).copy()
        * w[None, :, None]
    ).astype(np.complex64)
).to(
    signal.device
)  # (C, K, n_frames)

N_readout_effective = n_frames
coil_sensitivities_recon_tiled = torch.tile(
    coil_sensitivities_recon[:, :, :, None], (1, 1, 1, N_readout_effective)
)

In [ ]:
F = NonuniformFourierTransformOperator(
    k_traj_adc_reshaped, (N_channels, N_x, N_x, N_readout_effective)
)
C = ChannelOperator(coil_sensitivities_recon_tiled, (N_x, N_x, N_readout_effective))
A = F @ C
AHd = A.H @ data_reshaped.contiguous().view(torch.float32).ravel()
AHd_image = AHd.view(torch.complex64).reshape((N_x, N_x, N_readout_effective))

Q = A.H @ A
x = torch.zeros(N_x * N_x * N_readout_effective, dtype=torch.complex64).view(
    torch.float32
)
x_recon = conjugate_gradient(Q, AHd, x, maxiter=30)
x_recon_image = x_recon.view(torch.complex64).reshape((N_x, N_x, N_readout_effective))
print(f"reconstructed {N_readout_effective} frames, shape {tuple(x_recon_image.shape)}")

In [ ]:
b = AHd

In [ ]:
ximg = x_recon_image.cpu().numpy()  # (N_x, N_x, Nf) complex
Nf = ximg.shape[2]
n0 = N_max // 2
t_frames = centers * gre_block_dur * 1e3  # ms
phase = np.angle(ximg[:, :, -1])  # steady-state phase
signed = np.real(ximg * np.exp(-1j * phase)[:, :, None])
mag = np.abs(ximg)
print(
    f"frames: {Nf}, advance {frame_advance} TR -> {frame_advance*TR*1e3:.0f} ms; span {t_frames[0]:.0f}-{t_frames[-1]:.0f} ms"
)

ref = mag[:, :, -1]                # steady-state magnitude, same frame as reconstruction
thresh = 0.1 * ref.max()
brain = ref > thresh
tissues = [
    ("WM", brain & (T1m < 1.0), "tab:blue"),
    ("GM", brain & (T1m >= 1.0) & (T1m <= 2.0), "tab:orange"),
    ("CSF", brain & (T1m > 2.0), "tab:green"),
]

In [ ]:
vmax = 1.5 * float(np.percentile(mag, 95))

def den(i):
    lo, hi = max(0, i - 2), min(Nf, i + 3)
    return mag[:, :, lo:hi].mean(-1)


fnull = int(np.argmin(mag.reshape(-1, Nf)[brain.reshape(-1)].mean(0)))  # darkest frame
f_end = min(Nf - 1, int((1200.0 - t_frames[0]) / (frame_advance * TR * 1e3)))
frame_idx = np.sort(
    np.unique(np.concatenate([np.linspace(0, f_end, 11).round().astype(int), [fnull]]))
)
nrows, ncols = 3, (len(frame_idx) + 2) // 3
plt.figure(figsize=(ncols * 1.5, nrows * 1.5))
plt.suptitle("IR magnitude series (grayscale) - darkest frame = null")
for j, i in enumerate(frame_idx):
    ax = plt.subplot(nrows, ncols, j + 1)
    is_null = i == fnull
    plt.imshow(den(i).T, cmap="gray", vmin=0, vmax=vmax)
    plt.title(
        f"{t_frames[i]:.0f} ms",
        fontsize=9,
        color="red" if is_null else "k",
        fontweight="bold" if is_null else "normal",
    )
    for s_ in ax.spines.values():
        s_.set_edgecolor("red" if is_null else "0.8")
        s_.set_linewidth(2.5 if is_null else 0.5)
    plt.axis("off")
for j in range(len(frame_idx), nrows * ncols):
    plt.subplot(nrows, ncols, j + 1)
    plt.axis("off")
plt.show()

In [ ]:
class DeepADMM(nn.Module):

    def __init__(self, shape, rho_init=0.1, maxiter_admm=10, maxiter_cg=5):

        super().__init__()
        self.rho = nn.Parameter(torch.tensor(rho_init, dtype=torch.float32))
        self.regularizer = Regularizer(shape)
        self.I = IdentityOperator(shape)
        self.maxiter_admm = maxiter_admm
        self.maxiter_cg = maxiter_cg

    def forward(self,Q,b,x, disable_progressbar=True):
        z = torch.zeros(self.I.shape[0], dtype=x.dtype)
        u = torch.zeros(self.I.shape[0], dtype=x.dtype)

        for iteration in range(self.maxiter_admm):
    
            x = conjugate_gradient(
                Q + self.rho * self.I,
                b=b + self.rho * (z - u),
                x=x,
                maxiter=self.maxiter_cg,
                disable_progressbar=True,
            )
    
            z = self.regularizer(x + u)
            u = u + x - z
    
        return x

In [ ]:
sel = frame_idx
Nf = sel.size

k12 = k_traj_adc_reshaped[:, :, sel]
d12 = data_reshaped[:, :, sel]

F12 = NonuniformFourierTransformOperator(k12, (N_channels, N_x, N_x, Nf))
C12 = ChannelOperator(coil_sensitivities_recon_tiled[:, :, :, sel], (N_x, N_x, Nf))
A12 = F12 @ C12
Q12 = A12.H @ A12
b12 = A12.H @ d12.contiguous().view(torch.float32).ravel()

In [ ]:
shape = (N_x, N_x, 1)
torch.manual_seed(0)
model = DeepADMM(shape, rho_init=0.1, maxiter_admm=10, maxiter_cg=15)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# butterfly points per frame
Kb = k_traj_adc_reshaped. shape[1]
C1 = ChannelOperator(coil_sensitivities_recon, (N_x, N_x))
sel = frame_idx
num_iters = 100

model.train()
losses = []
for it in tqdm(range(num_iters), desc="Self-supervised splits"):
    np. random. seed(it)
    f = sel[np.random.randint(sel.size)]
    
    indices = np.arange(Kb)
    np.random. shuffle(indices)
    index_src, index_tar = np.array_split(indices, 2)
    
    k_src = k_traj_adc_reshaped[:, index_src, f][:, :, None]
    k_tar = k_traj_adc_reshaped[:, index_tar, f][:, :, None]
    d_src = data_reshaped[:, index_src, f].contiguous().view(torch.float32).ravel()
    d_tar = data_reshaped[:, index_tar, f].contiguous().view(torch.float32).ravel()
    
    F_src = NonuniformFourierTransformOperator(k_src, (N_channels, N_x, N_x, 1))
    F_tar = NonuniformFourierTransformOperator(k_tar, (N_channels, N_x, N_x, 1))
    
    Q_src = C1.H @ F_src.H @ F_src @ C1
    b_src = C1.H @ F_src.H @ d_src
    x_out = model(Q_src, b_src, b_src, disable_progressbar=True)
    k_data_pred = F_tar @ C1 @ x_out
    
    loss = torch. linalg.norm(k_data_pred - d_tar, ord=1) / torch.linalg.norm(d_tar, ord=1)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    tqdm.write(f"\rloss: {loss.item() :.4f}")

In [ ]:
model.eval()
recon = []
for f in range(Nf):
    k_f = k12[:, :, f][:, :, None]
    d_f = d12[:, :, f]
    F_f = NonuniformFourierTransformOperator(k_f, (N_channels, N_x, N_x, 1))
    C_f = ChannelOperator(coil_sensitivities_recon, (N_x, N_x))
    A_f= F_f@C_f
    Q_f = A_f.H@A_f
    b_f = A_f.H @ d_f.contiguous().view(torch. float32).ravel()
    with torch.no_grad():
        x_f = model(Q_f, b_f, b_f)
    recon.append(x_f.view(torch.complex64).reshape(N_x, N_x))

x_deep_admm_trained_image = torch.stack(recon, dim =- 1) # (N_x, N_x, Nf)

In [ ]:
t_sel = t_frames[sel]
vmax = 1.5 * float(np.percentile(np.abs(x_deep_admm_trained_image.cpu().numpy()), 95))

fig, axes = plt.subplots(3, 4, figsize=(7.5, 7.5))
for ax, i in zip(axes.ravel(), range(Nf)):
    ax.imshow(x_deep_admm_trained_image[:, :, i].abs().cpu().numpy().T, cmap="gray", vmin=0, vmax=vmax)
    ax.set_title(f"{t_sel[i] :.0f} ms", fontsize=9)
    ax.axis("off")
plt. suptitle("DeepADMM (learned, trained) - 12 frames")
plt. show()

In [ ]:
alpha_rad = np.deg2rad(alpha)  # 60° in your sequence params
cos_a2, sin_a2 = np.cos(alpha_rad / 2), np.sin(alpha_rad / 2)

T1_map = T1star * (S0 / S_StSt) * cos_a2
T2_map = T1star * sin_a2**2 / (1 - (S_StSt / S0) * cos_a2)
PD_map = S0 / sin_a2

T1_map = T1_map.reshape(N_x, N_x).cpu().numpy()
T2_map = T2_map.reshape(N_x, N_x).cpu().numpy()
PD_map = PD_map.reshape(N_x, N_x).cpu().numpy()
brain2d = brain.reshape(N_x, N_x)  # collapse the trailing singleton dim

for m in (T1_map, T2_map, PD_map):
    m[~brain2d] = 0

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))

im0 = axes[0].imshow(PD_map.T, cmap="gray", vmin=0, vmax=np.percentile(PD_map[brain2d], 99))
axes[0].set_title("Proton Density")
axes[0].axis("off")
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(T1_map.T * 1e3, cmap="hot", vmin=0, vmax=3000)  # ms
axes[1].set_title("T_1 [ms]")
axes[1].axis("off")
plt.colorbar(im1, ax=axes[1], fraction=0.046)

im2 = axes[2].imshow(T2_map.T * 1e3, cmap="hot", vmin=0, vmax=300)  # ms
axes[2].set_title("T_2 [ms]")
axes[2].axis("off")
plt.colorbar(im2, ax=axes[2], fraction=0.046)

plt.tight_layout()
plt.show()